In [3]:
# ============================================================
# CrimePulse_7Nation — Python EDA (single block)
# Loads all 4 clean fact tables + 3 dim tables from MySQL,
# merges into a master analysis dataframe, runs EDA across
# 9 sections, and saves every visual to its designated folder
# for later use in the Tableau dashboard build.
#
# Save this notebook file itself to:
# C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Source_Code\Python
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import URL, create_engine
import os

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 110

# ------------------------------------------------------------
# DB CONNECTION — replace placeholders with your real values.
# Use sqlalchemy.URL.create() so special characters in the
# password (@, #, $, !) are handled safely.
# ------------------------------------------------------------

DB_USER     = "Abishek"
DB_PASSWORD = "@b!$#3k@2003"
DB_HOST     = "localhost"
DB_PORT     = 3306
DB_NAME     = "CrimeRecordsDB"

connection_url = URL.create(
    "mysql+pymysql",
    username=DB_USER, password=DB_PASSWORD,
    host=DB_HOST, port=DB_PORT, database=DB_NAME,
)
engine = create_engine(connection_url)

# ------------------------------------------------------------
# OUTPUT FOLDERS (Visuals export paths — for Tableau prep)
# ------------------------------------------------------------
VIS = {
    "case_resolution":    r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Visuals\case_resolution",
    "crime_incident":     r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Visuals\crime_incident",
    "financial_digital":  r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Visuals\financial_digital",
    "victim_suspect":     r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Visuals\victim_suspect",
}
for path in VIS.values():
    os.makedirs(path, exist_ok=True)

def savefig(folder_key, filename):
    plt.tight_layout()
    plt.savefig(os.path.join(VIS[folder_key], filename), bbox_inches="tight")
    plt.close()

# ============================================================
# SECTION 1 — SETUP & DATA LOAD
# ============================================================
crime_incident    = pd.read_sql("SELECT * FROM crime_incident_fact_clean", engine)
victim_suspect    = pd.read_sql("SELECT * FROM victim_suspect_fact_clean", engine)
case_resolution   = pd.read_sql("SELECT * FROM case_resolution_fact_clean", engine)
financial_digital = pd.read_sql("SELECT * FROM financial_digital_fact_clean", engine)
dim_country       = pd.read_sql("SELECT * FROM dim_country", engine)
dim_crime_type    = pd.read_sql("SELECT * FROM dim_crime_type", engine)
dim_date          = pd.read_sql("SELECT * FROM dim_date", engine)

print("Rows loaded:")
for name, df in [("crime_incident", crime_incident), ("victim_suspect", victim_suspect),
                  ("case_resolution", case_resolution), ("financial_digital", financial_digital)]:
    print(f"  {name}: {len(df):,}")

# ============================================================
# SECTION 2 — MERGE INTO ANALYSIS-READY MASTER DATAFRAME
# ============================================================
df_master = (
    crime_incident
    .merge(victim_suspect, on="crime_id", how="left")
    .merge(case_resolution, on="crime_id", how="left")
    .merge(financial_digital, on="crime_id", how="left")
    .merge(dim_crime_type, left_on="crime_type", right_on="crime_type_name", how="left")
)
print(f"\ndf_master shape: {df_master.shape}  (expect 950,000 rows)")
assert len(df_master) == 950_000, "Row count mismatch after merge — check for join fan-out."

# ============================================================
# SECTION 3 — DATA QUALITY RECAP
# ============================================================
null_summary = df_master.isnull().mean().sort_values(ascending=False) * 100
null_summary = null_summary[null_summary > 0]
print("\nNull % by column (remaining, post-cleaning):")
print(null_summary)

plt.figure(figsize=(10, 5))
sns.heatmap(df_master.isnull(), cbar=False, yticklabels=False, cmap="rocket")
plt.title("Missingness Map — CrimePulse_7Nation (post-cleaning)")
savefig("crime_incident", "data_quality_missingness.png")

flag_cols = [
    "year_mismatch_flag", "severity_score_outlier_flag",
    "invalid_victim_count_flag", "victim_count_imputed_flag",
    "suspect_count_imputed_flag", "weapon_used_imputed_flag",
    "crime_status_imputed_flag", "invalid_duration_flag",
    "invalid_loss_flag", "financial_loss_imputed_flag",
]

print("\nAudit flag columns and their positive counts:")
for c in flag_cols:
    print(f"  {c}: {df_master[c].sum():,}")

# ============================================================
# SECTION 4 — UNIVARIATE EXPLORATION
# ============================================================
# Crime volume by year
plt.figure(figsize=(10, 5))
df_master.groupby("incident_year").size().plot(kind="line", marker="o")
plt.title("Total Crime Volume by Year (2010–2026)")
plt.xlabel("Year"); plt.ylabel("Number of Incidents")
savefig("crime_incident", "crime_volume_by_year.png")

# Crime type distribution
plt.figure(figsize=(10, 6))
df_master["crime_type"].value_counts().plot(kind="barh", color=sns.color_palette("mako", 11))
plt.title("Crime Type Distribution (Overall)")
plt.xlabel("Count")
savefig("crime_incident", "crime_type_distribution.png")

# Severity distribution
plt.figure(figsize=(7, 5))
order = ["Low", "Medium", "High", "Critical"]
sns.countplot(data=df_master, x="crime_severity", order=order, palette="rocket")
plt.title("Crime Severity Distribution")
savefig("crime_incident", "severity_distribution.png")

# Country crime volume
plt.figure(figsize=(8, 5))
df_master["country"].value_counts().plot(kind="bar", color=sns.color_palette("mako", 7))
plt.title("Total Crime Volume by Country")
plt.ylabel("Number of Incidents")
savefig("crime_incident", "country_crime_volume.png")

# weapon_used distribution
plt.figure(figsize=(8, 5))
victim_suspect["weapon_used"].value_counts().plot(kind="bar", color=sns.color_palette("crest", 6))
plt.title("Weapon Used Distribution")
savefig("victim_suspect", "weapon_used_distribution.png")

# victim_count distribution
plt.figure(figsize=(8, 5))
sns.histplot(victim_suspect["victim_count"], bins=25, kde=True, color="steelblue")
plt.title("Victim Count Distribution")
savefig("victim_suspect", "victim_count_distribution.png")

# suspect_count distribution
plt.figure(figsize=(8, 5))
sns.histplot(victim_suspect["suspect_count"], bins=21, kde=True, color="indianred")
plt.title("Suspect Count Distribution")
savefig("victim_suspect", "suspect_count_distribution.png")

# arrested_flag distribution
plt.figure(figsize=(6, 5))
sns.countplot(data=victim_suspect, x="arrested_flag", palette="rocket")
plt.title("Arrest Outcome Distribution")
savefig("victim_suspect", "arrested_flag_distribution.png")

# crime_status distribution
plt.figure(figsize=(8, 5))
case_resolution["crime_status"].value_counts().plot(kind="bar", color=sns.color_palette("mako", 5))
plt.title("Case Status Distribution")
savefig("case_resolution", "crime_status_distribution.png")

# ============================================================
# SECTION 5 — BIVARIATE / CROSS-CUT EXPLORATION
# ============================================================
# Crime type x country heatmap
pivot_ct_country = pd.crosstab(df_master["crime_type"], df_master["country"])
plt.figure(figsize=(10, 7))
sns.heatmap(pivot_ct_country, annot=True, fmt="d", cmap="rocket_r")
plt.title("Crime Type Distribution by Country")
savefig("crime_incident", "crimetype_country_heatmap.png")

# Severity x crime type
pivot_sev_ct = pd.crosstab(df_master["crime_type"], df_master["crime_severity"])
plt.figure(figsize=(9, 7))
sns.heatmap(pivot_sev_ct, annot=True, fmt="d", cmap="mako")
plt.title("Severity Distribution by Crime Type")
savefig("crime_incident", "severity_by_crimetype_heatmap.png")

# Resolution rate by country
res_by_country = (
    df_master.assign(is_resolved=df_master["crime_status"].isin(["Solved", "Closed"]))
    .groupby("country")["is_resolved"].mean() * 100
)
plt.figure(figsize=(8, 5))
res_by_country.sort_values().plot(kind="barh", color=sns.color_palette("crest", 7))
plt.title("Resolution Rate (%) by Country")
plt.xlabel("Resolution Rate (%)")
savefig("case_resolution", "resolution_rate_by_country.png")

# Arrest rate by crime type
arrest_by_type = (
    df_master.assign(is_arrested=df_master["arrested_flag"] == "Yes")
    .groupby("crime_type")["is_arrested"].mean() * 100
)
plt.figure(figsize=(9, 6))
arrest_by_type.sort_values().plot(kind="barh", color=sns.color_palette("rocket", 11))
plt.title("Arrest Rate (%) by Crime Type")
plt.xlabel("Arrest Rate (%)")
savefig("case_resolution", "arrest_rate_by_crime_type.png")

# Case duration distribution (excluding open/null cases)
plt.figure(figsize=(8, 5))
sns.histplot(case_resolution["case_duration_days"].dropna(), bins=30, kde=True, color="darkorange")
plt.title("Case Duration Distribution (Closed/Reported Cases Only)")
plt.xlabel("Days")
savefig("case_resolution", "case_duration_distribution.png")

# Financial loss by crime type (financial crime types only, boxplot)
financial_types = ["Embezzlement", "Currency Counterfeiting", "Capital Flight",
                    "Corporate Espionage", "Cyber Warfare", "Ransomware Extortion"]
df_fin = df_master[df_master["crime_type"].isin(financial_types)]
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_fin, x="crime_type", y="financial_loss_usd", palette="rocket")
plt.xticks(rotation=30, ha="right")
plt.title("Financial Loss Distribution by Crime Type (Financial Crimes Only)")
savefig("financial_digital", "financial_loss_by_crime_type.png")

# ============================================================
# SECTION 6 — GEOSPATIAL PREP
# ============================================================
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_master.sample(50_000, random_state=42),
                 x="longitude", y="latitude", hue="country", alpha=0.3, s=8, palette="tab10")
plt.title("Geospatial Distribution of Incidents by Country (sampled)")
savefig("crime_incident", "geospatial_scatter.png")

cb_by_country = df_master.groupby("country")["cross_border_flag"].apply(lambda x: (x == "Yes").mean() * 100)
plt.figure(figsize=(8, 5))
cb_by_country.sort_values().plot(kind="barh", color=sns.color_palette("mako", 7))
plt.title("Cross-Border Crime (%) by Country")
plt.xlabel("Cross-Border %")
savefig("financial_digital", "cross_border_by_country.png")

# ============================================================
# SECTION 7 — TIME SERIES DEEP DIVE
# ============================================================
monthly = df_master.groupby(["incident_year", "incident_month"]).size().reset_index(name="count")
monthly["year_month"] = monthly["incident_year"].astype(str) + "-" + monthly["incident_month"].astype(str).str.zfill(2)
plt.figure(figsize=(14, 5))
plt.plot(monthly["year_month"], monthly["count"], linewidth=0.8)
plt.xticks(rotation=90, fontsize=6)
plt.title("Monthly Crime Volume Trend (2010–2026)")
savefig("crime_incident", "monthly_seasonal_trend.png")

yearly = df_master.groupby("incident_year").size()
yoy = yearly.pct_change() * 100
plt.figure(figsize=(10, 5))
yoy.plot(kind="bar", color=np.where(yoy >= 0, "seagreen", "firebrick"))
plt.title("Year-over-Year % Change in Crime Volume")
plt.ylabel("% Change")
savefig("crime_incident", "yoy_change.png")

digital_trend = df_master.groupby("incident_year")["digital_crime_flag"].apply(lambda x: (x == "Yes").mean() * 100)
plt.figure(figsize=(10, 5))
digital_trend.plot(kind="line", marker="o", color="darkviolet")
plt.title("Digital Crime Share (%) Over Time")
plt.ylabel("% Digital Crime")
savefig("financial_digital", "digital_crime_trend.png")

# Financial loss overall distribution
plt.figure(figsize=(8, 5))
sns.histplot(financial_digital["financial_loss_usd"], bins=40, kde=True, color="teal")
plt.title("Financial Loss Distribution (All Records)")
savefig("financial_digital", "financial_loss_distribution.png")

# ============================================================
# SECTION 8 — EARLY KPI VALIDATION (printed, not saved)
# ============================================================
total_crimes = len(df_master)
resolution_rate = (df_master["crime_status"].isin(["Solved", "Closed"]).sum() / total_crimes) * 100
arrest_rate = (df_master["arrested_flag"] == "Yes").sum() / total_crimes * 100
avg_severity = df_master["severity_score"].mean()
total_financial_loss = df_master["financial_loss_usd"].sum()

print("\n--- EARLY KPI VALIDATION ---")
print(f"Total Crimes: {total_crimes:,}")
print(f"Resolution Rate: {resolution_rate:.2f}%")
print(f"Arrest Rate: {arrest_rate:.2f}%")
print(f"Average Severity Score: {avg_severity:.2f}")
print(f"Total Financial Loss (USD): ${total_financial_loss:,.2f}")

print("\nAll visuals exported successfully to the Visuals folders.")
print("EDA complete — ready for KPI calculation and Tableau dashboard build.")


Rows loaded:
  crime_incident: 950,000
  victim_suspect: 950,000
  case_resolution: 950,000
  financial_digital: 950,000

df_master shape: (950000, 38)  (expect 950,000 rows)

Null % by column (remaining, post-cleaning):
case_duration_days    4.959263
dtype: float64

Audit flag columns and their positive counts:
  year_mismatch_flag: 9,500
  severity_score_outlier_flag: 4,750
  invalid_victim_count_flag: 9,500
  victim_count_imputed_flag: 47,106
  suspect_count_imputed_flag: 38,000
  weapon_used_imputed_flag: 47,500
  crime_status_imputed_flag: 47,500
  invalid_duration_flag: 9,500
  invalid_loss_flag: 6,640
  financial_loss_imputed_flag: 34,939


C:\Users\ABISHEK.000\AppData\Local\Temp\ipykernel_26540\2558676469.py:132: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_master, x="crime_severity", order=order, palette="rocket")
C:\Users\ABISHEK.000\AppData\Local\Temp\ipykernel_26540\2558676469.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=victim_suspect, x="arrested_flag", palette="rocket")
C:\Users\ABISHEK.000\AppData\Local\Temp\ipykernel_26540\2558676469.py:224: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_fin, x="crime_type", y="financial_loss_usd", pa


--- EARLY KPI VALIDATION ---
Total Crimes: 950,000
Resolution Rate: 47.44%
Arrest Rate: 45.04%
Average Severity Score: 5.50
Total Financial Loss (USD): $30,213,662,852.52

All visuals exported successfully to the Visuals folders.
EDA complete — ready for KPI calculation and Tableau dashboard build.


In [4]:
# ============================================================
# SECTION 9 — EXPORT FINALIZED CSVs FOR TABLEAU (star schema)
# ============================================================
FINAL_PATH = r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Finalized"
os.makedirs(FINAL_PATH, exist_ok=True)

final_tables = {
    "crime_incident_final":     crime_incident,
    "victim_suspect_final":     victim_suspect,
    "case_resolution_final":    case_resolution,
    "financial_digital_final":  financial_digital,
    "dim_country_final":        dim_country,
    "dim_crime_type_final":     dim_crime_type,
    "dim_date_final":           dim_date,
}

print("Exporting finalized CSVs for Tableau:\n")
for name, df in final_tables.items():
    out_path = os.path.join(FINAL_PATH, f"{name}.csv")
    df.to_csv(out_path, index=False)
    size_mb = os.path.getsize(out_path) / (1024 * 1024)
    print(f"  {name}.csv — {len(df):,} rows, {size_mb:.1f} MB")

print(f"\nAll finalized tables exported to: {FINAL_PATH}")
print("Tableau: connect to all 7 CSVs and build relationships on crime_id (fact tables) and country/crime_type_name (dims).")

Exporting finalized CSVs for Tableau:

  crime_incident_final.csv — 950,000 rows, 92.3 MB
  victim_suspect_final.csv — 950,000 rows, 37.1 MB
  case_resolution_final.csv — 950,000 rows, 60.8 MB
  financial_digital_final.csv — 950,000 rows, 27.1 MB
  dim_country_final.csv — 7 rows, 0.0 MB
  dim_crime_type_final.csv — 11 rows, 0.0 MB
  dim_date_final.csv — 5,994 rows, 0.2 MB

All finalized tables exported to: C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Finalized
Tableau: connect to all 7 CSVs and build relationships on crime_id (fact tables) and country/crime_type_name (dims).
